# 01. UGV 로봇 개발 시작하기 (Getting Started)

영상 대응: **20:00–21:59**

> 3차 교육 영상 20–30분의 기능을 재구성한 **로컬 실습본**입니다. 공식 원본 노트북이 아닙니다. `mil_msgs` 원본 정의가 없어 이 환경에서는 `sim_mil_interfaces`를 사용합니다. 실제 ROS 2로 시뮬레이터와 통신합니다.

ROS 2는 로봇의 여러 프로그램이 데이터와 명령을 주고받도록 연결합니다.

| 개념 | 의미 | 이 실습의 예 |
|---|---|---|
| Node | ROS 2 통신에 참여하는 프로그램 단위 | 참가자 노드, 센서 브리지 |
| Topic | 데이터를 보내고 받는 통로 | `/mil/state`, `/image_raw`, `/cmd_vel` |
| Message | 토픽에서 주고받는 데이터 형식 | `MissionState`, `Twist`, `Image` |
| Service | 요청을 보내고 결과를 받는 방식 | `/mil/gimbal`, `/mil/fire` |

- **Subscribe**: 상태·경기 상태·카메라·IMU·위치를 수신합니다.
- **Publish**: 속도·OLED 문자열을 전송합니다. 별도 접수 응답은 없습니다.
- **Service**: 팬틸트·레이저 명령을 요청하고 응답을 확인합니다.
- **QoS**: 메시지 전달 방식입니다. 상태 토픽과 센서 토픽은 설정이 다릅니다.

현재 노트북은 Mac 안의 **Ubuntu 22.04 · ROS 2 Humble**에서 실행합니다. 로봇 하드웨어 대신 현재 3D 시뮬레이터에 연결합니다. Mac용 파일 편집·학습 커널과 구분합니다.

| 실습 | 기능 | 토픽/서비스 |
|---|---|---|
| 02.State | 체력·배터리·모드·경기 상태 | `/mil/state`, `/mil/match` |
| 03.Oled | 화면 4줄 문자열 | `/mil/oled` |
| 04.PanTilt | 좌우·상하 팬틸트 | `/mil/gimbal` |
| 05.Driving_Odometry | 속도 명령·출발점 기준 위치 | `/cmd_vel`, `/odom` |
| 06.Laser_Fire | 레이저 발사·대기시간 | `/mil/fire` |
| 07.Camera | 카메라 영상 한 장 | `/image_raw` |
| 08.Imu | 자세·위치 확인 | `/imu/data`, `/odom` |


## 실행 방법

위쪽 커널을 **Python 3 (ROS 2 · simulation)** 으로 선택하고, 아래 셀을 위에서부터 **Shift+Enter**로 실행합니다. 물리 경기장과 ROS 실습 연결을 먼저 켜세요. 각 실습은 시간이 지나면 종료되며, 셀을 다시 실행할 수 있습니다.


In [ ]:
import time
import math
import uuid
import rclpy
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy, qos_profile_sensor_data
from rclpy.signals import SignalHandlerOptions
from sim_mil_interfaces.msg import State as MissionState, Match as MatchCommand

# 상태/경기 토픽은 마지막 값을 새 구독자에게도 전달합니다.
STATUS_QOS = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                        durability=DurabilityPolicy.TRANSIENT_LOCAL)

def new_node(prefix):
    if not rclpy.ok():
        # Jupyter의 중단 버튼이 KeyboardInterrupt와 finally 정지를 처리하게 둡니다.
        rclpy.init(args=[], signal_handler_options=SignalHandlerOptions.NO)
    return rclpy.create_node(prefix + '_' + uuid.uuid4().hex[:8])

def spin_until(node, predicate, timeout=5.0):
    deadline = time.monotonic() + timeout
    while not predicate() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=min(0.05, max(0.0, deadline-time.monotonic())))
    if not predicate():
        raise TimeoutError('ROS 데이터/응답 시간 초과. 물리 경기장과 ROS 연결 상태를 확인하세요.')

def watch_status(node):
    data = {'state': None, 'match': None, 'state_at': None}
    def on_state(msg):
        data['state'] = msg
        data['state_at'] = time.monotonic()
    def on_match(msg):
        data['match'] = msg
    subscriptions = [
        node.create_subscription(MissionState, '/mil/state', on_state, STATUS_QOS),
        node.create_subscription(MatchCommand, '/mil/match', on_match, STATUS_QOS),
    ]
    spin_until(node, lambda: data['state'] is not None and data['match'] is not None)
    return data, subscriptions

def allowed(data):
    if data['state_at'] is None or time.monotonic()-data['state_at'] > 1.5:
        return False
    match = data['match']
    if match is None or match.cmd == 'pause':
        return False
    return match.mode == 'practice' or (match.mode == 'mission' and match.cmd in ('start', 'resume'))

def require_control(data):
    if not allowed(data):
        raise RuntimeError('제어 잠금 또는 상태 데이터 지연. 경기 재개·연결 상태를 확인하세요.')

def wait_publisher(node, publisher, timeout=5.0):
    spin_until(node, lambda: publisher.get_subscription_count() > 0, timeout)

def yaw_degrees(q):
    return math.degrees(math.atan2(2.0*(q.w*q.z + q.x*q.y), 1.0-2.0*(q.y*q.y+q.z*q.z)))

def source_age_seconds(node, message, maximum, label):
    stamp = message.header.stamp
    source_ns = int(stamp.sec)*1_000_000_000 + int(stamp.nanosec)
    if source_ns <= 0:
        raise ValueError(label + ': 원본 센서 시각이 없습니다.')
    age = (node.get_clock().now().nanoseconds-source_ns)/1_000_000_000
    if age < -0.25:
        raise ValueError(f'{label}: 센서 시각이 {-age:.3f}초 미래입니다. ROS 시계 설정을 확인하세요.')
    if age > maximum:
        raise TimeoutError(f'{label}: 원본 데이터가 {age:.3f}초 전 값입니다.')
    return age


## 실습 설정

아래 값을 바꾸고 다시 실행할 수 있습니다.


In [ ]:
DISCOVERY_SECONDS = 2.0


## 실행


In [ ]:
import os
import platform
from IPython.display import display, Markdown
node = new_node('demo_getting_started')
try:
    end = time.monotonic() + DISCOVERY_SECONDS
    while time.monotonic() < end:
        rclpy.spin_once(node, timeout_sec=0.05)
    topics = dict(node.get_topic_names_and_types())
    services = dict(node.get_service_names_and_types())
    expected_topics = ['/mil/state', '/mil/match', '/mil/oled', '/cmd_vel', '/odom', '/image_raw', '/imu/data']
    expected_services = ['/mil/gimbal', '/mil/fire']
    print('OS:', platform.system(), '| ROS:', os.environ.get('ROS_DISTRO'), '| Python:', platform.python_version())
    print('노트북 위치:', os.getcwd())
    rows = ['| 경로 | 실제 ROS 타입 |', '|---|---|']
    for topic in expected_topics:
        rows.append('| `' + topic + '` | ' + ', '.join(topics.get(topic, ['미발견'])) + ' |')
    for service in expected_services:
        rows.append('| `' + service + '` | ' + ', '.join(services.get(service, ['미발견'])) + ' |')
    display(Markdown('\n'.join(rows)))
    missing = [t for t in expected_topics if t not in topics] + [s for s in expected_services if s not in services]
    DEMO_RESULT = {'demo':'01', 'ros_distro':os.environ.get('ROS_DISTRO'), 'topics':topics, 'services':services, 'missing':missing}
    if missing:
        print('연결 확인 필요:', ', '.join(missing))
    else:
        print('실습용 ROS 토픽·서비스 연결을 확인했습니다.')
finally:
    node.destroy_node()


## 코드에서 연결되는 과정

`노트북의 rclpy 노드 → ROS 2 토픽/서비스 → 로봇 브리지 → 시뮬레이터`

공식 영상의 `mil_msgs/msg/MissionState`, `mil_msgs/msg/MatchCommand`는 이 실습에서 `sim_mil_interfaces/msg/State`, `sim_mil_interfaces/msg/Match`로 대응합니다. 코드 안의 이름을 `as MissionState`, `as MatchCommand`로 맞췄으며, 공식 패키지를 설치한 것으로 표시하지 않습니다.

JupyterLab은 코드를 쓰고 실행하는 작업 도구입니다. 노트북 실행 자체가 인식 모델 학습은 아닙니다.


[← 실습 목차](00.실습목차.ipynb) · [코드 편집](../team_node.py)

실물 장치의 토픽 타입·주기·응답과 맞는지는 지급 UGV에서 확인해야 합니다.
